<a href="https://colab.research.google.com/github/Mohsin-22/Assignment5-100_Gen-Ai_cohort/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

# Assignment 5: Build a Support Ticket Classifier using Pre-trained Language Models from Hugging Face

In [ ]:
!pip install transformers datasets torch scikit-learn pandas

In [ ]:
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
dataset = load_dataset("Tobi-Bueck/customer-support-tickets")

In [ ]:
print(dataset)


In [ ]:
split_data = dataset["train"].train_test_split(test_size=0.2, seed=42)

train_data = split_data["train"]
test_data = split_data["test"]

In [ ]:

text_columns = ["subject", "body"]
label_column = ""


In [ ]:
def merge_text_fields(example):
    subject = example["subject"] if example["subject"] is not None else ""
    body = example["body"] if example["body"] is not None else ""
    example["text"] = subject + " " + body
    return example



In [ ]:

train_data = train_data.map(merge_text_fields)
test_data = test_data.map(merge_text_fields)


In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [ ]:
train_data = train_data.map(tokenize_batch, batched=True)
test_data = test_data.map(tokenize_batch, batched=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

train_labels = label_encoder.fit_transform(train_data["type"])
test_labels = label_encoder.transform(test_data["type"])

num_labels = len(label_encoder.classes_)
print("Ticket classes:", label_encoder.classes_)

In [ ]:
columns_to_remove = [
    "subject", "body", "answer", "type",
    "queue", "priority", "language", "version",
    "tag_1", "tag_2", "tag_3", "tag_4",
    "tag_5", "tag_6", "tag_7", "tag_8"
]

In [ ]:
train_data = train_data.remove_columns(columns_to_remove)
test_data = test_data.remove_columns(columns_to_remove)


In [ ]:
train_data = train_data.add_column("labels", train_labels)
test_data = test_data.add_column("labels", test_labels)


In [ ]:
train_data.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_data.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, predictions)}

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./support_ticket_classifier",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    report_to="none"
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./support_ticket_classifier",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    report_to="none",
    optim="adamw_torch"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics
)

trainer.train()


In [ ]:
model_cpu = model.to("cpu")
model_cpu.eval()


In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(all_labels, all_preds)
print("Test Accuracy:", accuracy)

In [ ]:
def classify_ticket(subject, body):
    text = subject + " " + body
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    predicted_index = torch.argmax(outputs.logits, dim=1).item()
    return label_encoder.inverse_transform([predicted_index])[0]

In [ ]:
classify_ticket(
    "Refund not processed",
 "I applied for a refund two weeks ago but it is still pending."
)

In [ ]:
print(label_encoder.classes_)

In [ ]:
!pip install transformers datasets torch scikit-learn pandas

In [ ]:
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
dataset = load_dataset("Tobi-Bueck/customer-support-tickets")

In [ ]:
print(dataset)


In [ ]:
split_data = dataset["train"].train_test_split(test_size=0.2, seed=42)

train_data = split_data["train"]
test_data = split_data["test"]

In [ ]:

text_columns = ["subject", "body"]
label_column = ""


In [ ]:
def merge_text_fields(example):
    subject = example["subject"] if example["subject"] is not None else ""
    body = example["body"] if example["body"] is not None else ""
    example["text"] = subject + " " + body
    return example



In [ ]:

train_data = train_data.map(merge_text_fields)
test_data = test_data.map(merge_text_fields)


In [ ]:
from transformers import AutoTokenizer

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [ ]:
train_data = train_data.map(tokenize_batch, batched=True)
test_data = test_data.map(tokenize_batch, batched=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

train_labels = label_encoder.fit_transform(train_data["type"])
test_labels = label_encoder.transform(test_data["type"])

num_labels = len(label_encoder.classes_)
print("Ticket classes:", label_encoder.classes_)

In [ ]:
columns_to_remove = [
    "subject", "body", "answer", "type",
    "queue", "priority", "language", "version",
    "tag_1", "tag_2", "tag_3", "tag_4",
    "tag_5", "tag_6", "tag_7", "tag_8"
]

In [ ]:
train_data = train_data.remove_columns(columns_to_remove)
test_data = test_data.remove_columns(columns_to_remove)


In [ ]:
train_data = train_data.add_column("labels", train_labels)
test_data = test_data.add_column("labels", test_labels)


In [ ]:
train_data.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

test_data.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels, predictions)}

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./support_ticket_classifier",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    report_to="none"
)

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./support_ticket_classifier",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    report_to="none",
    optim="adamw_torch"
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_metrics
)

trainer.train()


In [ ]:
model_cpu = model.to("cpu")
model_cpu.eval()


In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(all_labels, all_preds)
print("Test Accuracy:", accuracy)

In [ ]:
def classify_ticket(subject, body):
    text = subject + " " + body
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    predicted_index = torch.argmax(outputs.logits, dim=1).item()
    return label_encoder.inverse_transform([predicted_index])[0]

In [ ]:
classify_ticket(
    "Refund not processed",
 "I applied for a refund two weeks ago but it is still pending."
)

In [ ]:
print(label_encoder.classes_)